In [8]:
import pandas as pd
import os
import glob

# Find the step export in the repository raw-data directory.
try:
    data_dir = '../../data/raw/samsung_health/export_20260516'
    # Corrected the filename pattern to match the actual file   com.samsung.shealth.step_daily_trend.*.csv
    step_files = glob.glob(os.path.join(data_dir, 'com.samsung.shealth.step_daily_trend.*.csv'))
    if not step_files:
        raise FileNotFoundError(f"No daily-step CSV file found in {data_dir}.")
    INPUT_FILE = step_files[0]  # Use the first match
    print(f"Found input file: {INPUT_FILE}")
except FileNotFoundError as e:
    print(e)
    INPUT_FILE = None

OUTPUT_FILE = "../../data/processed/daily_steps_summary.csv"

# Updated column names based on the new file type
DAY_TIME_COL = "day_time"
# DATAUUID_COL is not a reliable time source and can cause overflow, so it's removed from candidates.
# DATAUUID_COL = "datauuid" 
CREATE_TIME_COL = "create_time"
UPDATE_TIME_COL = "update_time"
COUNT_COL = "step_count" # The column is likely named 'step_count' in this file
SOURCE_TYPE_COL = "source_type"


def build_daily_summary(input_file: str = INPUT_FILE, output_file: str = OUTPUT_FILE) -> pd.DataFrame:
    if not input_file or not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found or is invalid: {input_file}")

    # فایل خروجی Samsung Health یک سطر متادیتا قبل از هدر اصلی دارد.
    df = pd.read_csv(input_file, skiprows=1, dtype=str, low_memory=False)

    # --- تشخیص ستون تاریخ روز ---
    date_candidates: dict[str, pd.Series] = {}
    date_scores: dict[str, int] = {}

    # Only use day_time for epoch conversion to avoid FloatingPointError from large UUIDs.
    for col in [DAY_TIME_COL]:
        if col in df.columns:
            epoch_numeric = pd.to_numeric(df[col], errors="coerce")
            
            # Safeguard against overflow: filter out impossibly large numbers before conversion.
            # A 15-digit number in milliseconds is far in the future, indicating an invalid value.
            epoch_numeric = epoch_numeric[epoch_numeric < 1e15] 
            
            epoch_dates = pd.to_datetime(epoch_numeric, unit="ms", errors="coerce")
            date_candidates[col] = epoch_dates
            date_scores[col] = int(epoch_dates.notna().sum())

    if max(date_scores.values(), default=0) == 0:
        for col in [CREATE_TIME_COL, UPDATE_TIME_COL]:
            if col in df.columns:
                parsed = pd.to_datetime(df[col], errors="coerce")
                date_candidates[col] = parsed
                date_scores[col] = int(parsed.notna().sum())

    if not date_scores:
        raise ValueError("No date/time columns found in this step dataset.")

    selected_date_col = max(date_scores, key=date_scores.get)
    if date_scores[selected_date_col] == 0:
        raise ValueError("No parseable date values found in candidate date columns.")

    # --- تشخیص ستون مقدار گام (با پوشش حالت شیفت ستون‌ها) ---
    step_candidate_cols = [COUNT_COL, SOURCE_TYPE_COL]
    step_scores: dict[str, tuple[int, float, float]] = {}
    step_series_map: dict[str, pd.Series] = {}

    for col in step_candidate_cols:
        if col not in df.columns:
            continue

        numeric_values = pd.to_numeric(df[col], errors="coerce")
        valid = numeric_values.dropna()
        if valid.empty:
            continue

        integer_like_ratio = ((valid % 1) == 0).mean()
        large_enough_ratio = (valid >= 10).mean()
        score = int(valid.count())
        step_scores[col] = (score, float(integer_like_ratio), float(large_enough_ratio))
        step_series_map[col] = numeric_values

    if not step_scores:
        raise ValueError("Could not find a numeric step-count column.")

    selected_step_col = max(step_scores, key=lambda key: step_scores[key])
    step_values = step_series_map[selected_step_col]

    clean_df = pd.DataFrame(
        {
            "date": date_candidates[selected_date_col].dt.date,
            "step_count": step_values,
        }
    )

    clean_df = clean_df.dropna(subset=["date", "step_count"])

    # برای هر روز فقط یک مقدار قدم نگه می‌داریم (بیشترین مقدار ثبت‌شده‌ی روز)
    daily_summary = (
        clean_df.groupby("date", as_index=False)["step_count"]
        .max()
        .rename(columns={"step_count": "daily_step_count"})
        .sort_values("date")
    )

    daily_summary["daily_step_count"] = daily_summary["daily_step_count"].round().astype("Int64")

    daily_summary.to_csv(output_file, index=False, encoding="utf-8-sig")
    return daily_summary


summary = build_daily_summary()
print(summary.head(15).to_string(index=False))
print(f"\nSaved: {OUTPUT_FILE}")
print(f"Rows: {len(summary)}")

Found input file: New Data\com.samsung.shealth.step_daily_trend.20260516153602.csv
      date  daily_step_count
2026-02-09              2944
2026-02-10              3751
2026-02-11              3751
2026-02-12              3729
2026-02-13              4038
2026-02-14              4038
2026-02-15              6760
2026-02-16              6760
2026-02-17              4073
2026-02-18              4073
2026-02-19              4943
2026-02-20             13535
2026-02-21             13535
2026-02-22              3893
2026-02-23              3210

Saved: daily_steps_summary.csv
Rows: 96
